In [1]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import numpy as np
import matplotlib.pyplot as plt
import qutip
from qblox_scheduler import Schedule, HardwareAgent
from qblox_scheduler.operations import GaussPulse, SquarePulse, DRAGPulse, Measure
from qblox_scheduler.resources import ClockResource
from qblox_sim.simulator import QbloxQutipSimulator

%matplotlib inline

/opt/anaconda3/envs/blox/lib/python3.14/site-packages/quantify_core/utilities/general.py:13: QCoDeSDeprecationWarning: The `qcodes.utils.helpers` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.utils.helpers import NumpyJSONEncoder


## 2. Defining the Qubit Model

First, we define the physical properties of our qubit. The simulator uses these parameters to construct the Hamiltonian and collapse operators ($T_1$, $T_2$).

In [2]:
qubit_params = {
    'f_q': 5.0e9,           # Qubit frequency: 5 GHz
    'f_d': 5.0e9,           # Drive frequency (on resonance)
    'rabi_freq_per_volt': 25e6, # 25 MHz per Volt drive strength
    'T1': 2e-6,            # 2 us relaxation time
    'T2': 1e-6,            # 1 us dephasing time
}
hardware_config_path = "config_files/hw_config.json"
device_config_path = "config_files/dut_config.json"

agent = HardwareAgent(hardware_config_path, device_config_path)

configs = {"hardware": hardware_config_path, "device": device_config_path}
sim = QbloxQutipSimulator(qubit_params, configs)
print("Simulator initialized with qubit frequency:", qubit_params['f_q'] / 1e9, "GHz")

Simulator initialized with qubit frequency: 5.0 GHz


## 3. Creating a Schedule

We use `qblox-scheduler` to define a simple experiment: a Rabi pulse followed by a measurement.

In [3]:
sched = Schedule("Rabi Tutorial")
sched.add_resource(ClockResource(name="q0.01", freq=30e6))
sched.add_resource(ClockResource(name="q0.ro", freq=60e6))

# Add a Gauss pulse
sched.add(GaussPulse(amplitude=0.5, phase=0, duration=80e-9, port="q0:mw", clock="q0.01"))
pulse = sched.add(GaussPulse(amplitude=0.5, phase=0, duration=80e-9, port="q0:mw", clock="q0.01"))

# Add a measurement 10ns after the pulse
sched.add(Measure("q0"), rel_time=8e-9, ref_op=pulse, ref_pt="end")

agent.run(sched)

print("Schedule created.")

/opt/anaconda3/envs/blox/lib/python3.14/site-packages/qblox_scheduler/qblox/hardware_agent.py:499: UserWarning: cluster: Trying to instantiate cluster with ip 'None'.Creating a dummy cluster.
  warnings.warn(
/opt/anaconda3/envs/blox/lib/python3.14/site-packages/qblox_scheduler/backends/circuit_to_device.py:440: RuntimeWarning: Clock 'q0.01' has conflicting frequency definitions: 30000000.0 Hz in the schedule and 50000000.0 Hz in the device config. The clock is set to '30000000.0'. Ensure the schedule clock resource matches the device config clock frequency or set the clock frequency in the device config to np.NaN to omit this warning.
  warnings.warn(
/opt/anaconda3/envs/blox/lib/python3.14/site-packages/qblox_scheduler/backends/circuit_to_device.py:440: RuntimeWarning: Clock 'q0.ro' has conflicting frequency definitions: 60000000.0 Hz in the schedule and 100000000.0 Hz in the device config. The clock is set to '60000000.0'. Ensure the schedule clock resource matches the device config

RuntimeError: ('Assembly failed.', "setting cluster_module2_sequencer0_sequence to {'program': ' set_mrk 0 # set markers to 0 (init)\\n wait_sync 4 \\n upd_param 4 \\n wait 4 # latency correction of 4 + 0 ns\\n move 1,R0 # iterator for loop with label start\\nstart:   \\n reset_ph  \\n upd_param 4 \\n set_awg_gain 16377,0 # setting gain for GaussPulse\\n play 0,0,4 # play GaussPulse (80 ns)\\n wait 76 # auto generated wait (76 ns)\\n set_awg_gain 16377,0 # setting gain for GaussPulse\\n play 0,0,4 # play GaussPulse (80 ns)\\n wait 5284 # auto generated wait (5284 ns)\\n loop R0,@start \\n stop  \\n', 'waveforms': {'2275612660640677631': {'data': [-8.128026702117856e-05, 8.128026702117991e-05, 0.00031522475117620453, 0.0006483273758335009, 0.001117567355024124, 0.001771520301977271, 0.002673120296885708, 0.0039027572971171163, 0.005561631452321304, 0.00777523199901434, 0.010696744943692387, 0.014510123543795203, 0.019432483410877114, 0.025715416675417997, 0.03364476578743409, 0.04353836

In [ ]:
# 1. Compile the schedule
compiled_sched = agent.compile(sched)

# 2. Inspect the top-level keys in compiled_instructions
print("Available instruments:", list(compiled_sched.compiled_instructions.keys()))

Qblox Scheduler is a compiler for Q1ASM, so the sequence program can be extracted from a Schedule object:

In [ ]:
comp = agent.compile(sched)
comp.compiled_instructions

## Setup virtual Qblox Cluster

In [ ]:
from q1simulator import Q1Simulator
from q1simulator import Cluster

modules = {
    2: "QCM_RF",
    4: "QRM_RF",
    }

cluster = ('cluster', modules)


In [ ]:
from q1simulator import Q1Plotter

cluster = agent.get_clusters()["cluster"]

plotter = Q1Plotter(cluster)
plotter.plot(t_max = 500)

In [ ]:
dat_0 = plotter._simulator.get_connected_modules()[2].get_output()['sequencer0-I'].data

fig,ax = plt.subplots()

ax.plot(dat_0[:500])
plotter._simulator.get_simulation_end_time()

In [ ]:
qcm = plotter._simulator.get_connected_modules()[2]
qcm.arm_sequencer(0)
qcm.get_output()['sequencer0-I']

In [ ]:
qrm = qcm = plotter._simulator.get_connected_modules()[4]
list(qrm.get_acquisition_windows().keys())[0]

## 4. Running the Simulation

The `simulate()` method perform the following:
1. Resolves absolute timing of all operations.
2. Extracts pulse envelopes (I/Q) over time.
3. Constructs the time-dependent Hamiltonian $H(t)$.
4. Solves the Lindblad master equation using QuTiP's `mesolve`.

In [ ]:
results = sim.simulate(sched)

t_list = results['t_list']
qubit_states = results['result'].states

# Calculate expectation values for Bloch sphere coordinates
expt_x = [qutip.expect(sim.sx, s).real for s in qubit_states]
expt_y = [qutip.expect(sim.sy, s).real for s in qubit_states]
expt_z = [qutip.expect(sim.sz, s).real for s in qubit_states]

plt.figure(figsize=(10, 5))
#plt.plot(t_list[:200] * 1e9, expt_x[:200], label='X')
#plt.plot(t_list[:200] * 1e9, expt_y[:200], label='Y')
plt.plot(t_list * 1e9, expt_z, label='Z')
plt.axvline(x=40, color='gray', linestyle='--', label='Pulse End')
plt.xlabel('Time (ns)')
plt.ylabel('Expectation Value')
plt.title('Bloch Vector Evolution during Gaussian Pulse')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
results

In [ ]:
val = 1/np.sqrt(2)

meas_state = results['measurements'][0]['outcome']
I_Q = np.array([val * meas_state, val * 1j * meas_state])

In [ ]:
mean = 0
sigma = 0.1

noise = np.random.normal(mean, sigma, I_Q.shape)
I_Q_noisy = I_Q + noise
print("Noisy I/Q data:", I_Q_noisy)

In [ ]:
from q1simulator import Cluster

modules = {
    2: "QCM",
    4: "QRM",
    }

cluster = Cluster(name='cluster', modules=modules)
qcm = cluster.get_connected_modules(2)

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import numpy as np
import scipy

from qblox_sim.simulator import QbloxQ1Simulator


from q1simulator import Cluster

modules = {
    2: "QCM",
    4: "QRM",
    }


hardware_config = {"drive": {"module":2, "sequencer":0}, "readout": {"module":4, "sequencer":0}}

detuning = 0 

qubit_params = {
    'f_q': 5.0e9,           # Qubit frequency: 5 GHz
    'f_d': 5.0e9 - detuning,           # Drive frequency (on resonance)
    'f_r': 6.0e9,           # Readout frequency
    'rabi_freq_per_volt': 25e6, # 250 kHz per Volt drive strength
    'T1': 200e-6,            # 200 us relaxation time
    'T2': 100e-6,            # 100 us dephasing time
}

sim = QbloxQ1Simulator(qubit_params, 'cluster0', modules, hardware_config)

# Access the cluster through the simulator
qcm = sim.cluster.get_connected_modules()[2]
qrm = sim.cluster.get_connected_modules()[4]

In [ ]:
pulse_length = 30 #in ns
tof = 100 #time of flight in ns
drive_freq = 100e6
readout_freq = 6.0e7

gaussian_wave = scipy.signal.windows.gaussian(pulse_length, std=0.12 * pulse_length).tolist()
square_wave = np.ones(pulse_length).tolist()


control_program = f"""
        # --- Q1 Core Real-Time Assembly (Q1ASM) ---

        # 1. Synchronize! Wait for the global hardware clock pulse
        wait_sync 4

        wait    400

        # 2. Set AWG gain to maximum (32767 is max for 16-bit Qblox hardware)
        set_awg_gain 16000, 16000

        # 3. Play the VQE rotation pulse (I=wave 0, Q=wave 1)
        play    1, 1, {pulse_length}

        # 4. Wait
        wait 100


        # 5. Halt the sequencer
        stop
    """


readout_program = f"""
        # --- Q1 Core Real-Time Assembly (Q1ASM) ---

        # 1. Synchronize! Wait for the global hardware clock pulse
        wait_sync 4

        acquire 0, 1, 400

        # 2. Set AWG gain to maximum (32767 is max for 16-bit Qblox hardware)
        set_awg_gain 32000, 32000

        # 3. Play the VQE rotation pulse (I=wave 0, Q=wave 1)
        play    1, 1, 60

        # 4. Wait for the time of flight
        wait {tof}

        # 5. Trigger the readout acquisition (Acq Index 0, Bin 0)
        acquire 0, 0, 400

        # 6. Halt the sequencer
        stop
    """

readout_q1asm_sequence = {
    "waveforms": {
        "vqe_rx_I": {"data": gaussian_wave, "index": 0},
        "vqe_rx_Q": {"data": square_wave, "index": 1}
    },
    "weights": {},
    "acquisitions": {
        "vqe_readout": {"num_bins": 2, "index": 0}
    },
    "program": readout_program
}

control_q1asm_sequence = {
    "waveforms": {
        "vqe_tx_I": {"data": gaussian_wave, "index": 0},
        "vqe_tx_Q": {"data": square_wave, "index": 1}
    },
    "weights": {},
    "acquisitions": {},
    "program": control_program
}


In [ ]:
# Virtual Cabling
qrm.sequencer0.connect_out0('I')
qrm.sequencer0.connect_out1('Q')

qcm.sequencer0.connect_out0('I')
qcm.sequencer0.connect_out1('Q')


# --- THE CRITICAL FIXES ---
# Opt the sequencer into the global synchronization group
qrm.sequencer0.sync_en(True)
# Turn up the master AWG gain for the I and Q paths
qrm.sequencer0.gain_awg_path0(1.0)
qrm.sequencer0.gain_awg_path1(1.0)

qrm.sequencer0.mod_en_awg(True)
qrm.sequencer0.demod_en_acq(True)

# set resonator readout NCO freq
qrm.sequencer0.nco_freq(readout_freq)

# enable and set LO frequency
# qrm.parameters['out0_lo_en'] = True
# qrm.parameters['out0_lo_freq'] = qubit_params['f_r'] - readout_freq

# Opt the sequencer into the global synchronization group
qcm.sequencer0.sync_en(True)
# Turn up the master AWG gain for the I and Q paths
qcm.sequencer0.gain_awg_path0(1.0)
qcm.sequencer0.gain_awg_path1(1.0)

qcm.sequencer0.mod_en_awg(True)

# set MW drive NCO frequency
qcm.sequencer0.nco_freq(drive_freq)

# enable and set LO frequency

# qcm.parameters['out0_lo_en'] = True
# qcm.parameters['out0_lo_freq'] = qubit_params['f_d'] - drive_freq


In [ ]:
# Load and execute
qcm.sequencer0.sequence(control_q1asm_sequence)

qcm.arm_sequencer(0)

qrm.sequencer0.sequence(readout_q1asm_sequence)
qrm.arm_sequencer(0)

print("Executing real-time Q1ASM on virtual FPGA...")


sim.cluster.start_sequencer()

In [ ]:
from q1simulator import Q1Plotter

plotter = Q1Plotter(sim.cluster)
plotter.plot(t_max=500)

In [ ]:
qcm.get_output()['sequencer0-Q']

In [ ]:
results = sim.simulate()

In [ ]:
#results = sim.simulate()

import matplotlib.pyplot as plt
import qutip

t_list = results['t_list']
qubit_states = results['result'].states

# Calculate expectation values for Bloch sphere coordinates
expt_z = [qutip.expect(sim.sz, s).real for s in qubit_states]



plt.figure(figsize=(10, 5))
plt.plot(t_list * 1e9, expt_z, label='Z')
plt.axvline(x=40, color='gray', linestyle='--', label='Pulse End')
plt.xlabel('Time (ns)')
plt.ylabel('Expectation Value')
plt.title('Bloch Vector Evolution during Gaussian Pulse')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
measurements = results['measurements']
measurements

In [ ]:
I = np.array([m['I'] for m in measurements])
Q = np.array([m['Q'] for m in measurements])

In [ ]:
qrm.sequencers[0].set_acquisition_mock_data([5,5])

In [ ]:
qrm.get_acquisitions(0)['vqe_readout']['acquisition']['bins']['integration']

In [ ]:
acq_windows=qrm.get_acquisition_windows()

In [ ]:
t_list = np.linspace(0, 35, 35)
measurements = []

mean = 0
sigma = 0.1 
for acq in acq_windows['sequencer0']:
    print(acq)
    try:
        acq_start = acq[0][0]
        idx = np.argmin(np.abs(t_list - acq_start))
        state = results['result'].states[idx]
        rho_q = state.ptrace(0) if state.type == 'oper' else qutip.ket2dm(state).ptrace(0)
        prob_1 = np.real(qutip.expect(qutip.ket2dm(qutip.basis(2, 1)), rho_q))
        I = 1/np.sqrt(2)*prob_1 * np.random.normal(mean, sigma, 1)
        Q = 1j*1/np.sqrt(2)*prob_1 * np.random.normal(mean, sigma, 1)
        measurements.append({
            'time': acq_start, 'prob_1': prob_1,
            'outcome': 1 if np.random.random() < prob_1 else 0,
            'I': I,
            'Q': Q
        })
    except Exception as e:
        print(f"Warning: Could not process acquisition: {e}")
        continue

In [ ]:
measurements

In [ ]:
expt_z = [qutip.expect(sim.sz, s).real for s in qubit_states]
expt_z

In [ ]:
results['result'].states

In [ ]:
rho_q = state.ptrace(0) if state.type == 'oper' else qutip.ket2dm(state).ptrace(0)
prob_1 = np.real(qutip.expect(qutip.ket2dm(qutip.basis(2, 1)), rho_q))